# 21 — Incremental Loading with Watermarks

Demonstrates `DataHelper.load_incremental()` for delta-loading from SQL sources.

Key APIs showcased:
- `FileWatermarkStore` — JSON-backed persistence of watermark state
- `DataHelper.load_incremental()` / `.aload_incremental()` — sync and async delta load
- pandas, dask, and polars engine views all support incremental loading
- `IncrementalResult` — `records_loaded`, `previous_watermark`, `current_watermark`, `watermark_committed`

In [1]:
import datetime as dt
import os
from pathlib import Path
from tempfile import TemporaryDirectory

from sqlalchemy import Date, Integer, String, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column

from boti_data import (
    DataHelper,
    FileWatermarkStore,
    IncrementalResult,
)

In [2]:
class Base(DeclarativeBase):
    pass


class Event(Base):
    __tablename__ = "events"

    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    event_date: Mapped[dt.date] = mapped_column(Date())
    status: Mapped[str] = mapped_column(String(16))

In [3]:
tmp = TemporaryDirectory()
root = Path(tmp.name)
db_path = root / "events.db"
sqlite_dsn = f"sqlite:///{db_path}"
watermark_path = root / "watermarks.json"

engine = create_engine(sqlite_dsn)
Base.metadata.create_all(engine)
with Session(engine) as session:
    session.add_all([
        Event(id=1, event_date=dt.date(2026, 5, 1), status="active"),
        Event(id=2, event_date=dt.date(2026, 5, 2), status="inactive"),
        Event(id=3, event_date=dt.date(2026, 5, 3), status="active"),
    ])
    session.commit()
engine.dispose()

watermark_store = FileWatermarkStore(str(watermark_path))
print(f"Seeded 3 rows at {sqlite_dsn}")

Seeded 3 rows at sqlite:////var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/tmpu8il3px6/events.db


## First run — full load

No watermark exists yet, so `load_incremental()` performs a full load and commits the max `event_date` as the watermark.

In [4]:
with DataHelper(
    backend="sqlalchemy",
    connection_url=sqlite_dsn,
    poolclass="sqlalchemy.pool.NullPool",
    query_only=False,
    table="events",
) as helper:
    result = helper.pandas.load_incremental(
        watermark_field="event_date",
        watermark_source="events_table",
        watermark_store=watermark_store,
    )

print(f"records_loaded:      {result.records_loaded}")
print(f"previous_watermark:  {result.previous_watermark}")
print(f"current_watermark:   {result.current_watermark}")
print(f"watermark_committed: {result.watermark_committed}")
result.frame

records_loaded:      3
previous_watermark:  None
current_watermark:   2026-05-03 00:00:00+00:00
watermark_committed: True


,id,event_date,status
0,1,2026-05-01 00:00:00+00:00,active
1,2,2026-05-02 00:00:00+00:00,inactive
2,3,2026-05-03 00:00:00+00:00,active


## Second run — no new data

All rows are at or before the stored watermark, so the result is empty.

In [5]:
with DataHelper(
    backend="sqlalchemy",
    connection_url=sqlite_dsn,
    poolclass="sqlalchemy.pool.NullPool",
    query_only=False,
    table="events",
) as helper:
    result = helper.pandas.load_incremental(
        watermark_field="event_date",
        watermark_source="events_table",
        watermark_store=watermark_store,
    )

print(f"records_loaded:      {result.records_loaded}")
print(f"current_watermark:   {result.current_watermark}")
print(f"watermark_committed: {result.watermark_committed}")
print(f"bool(result):        {bool(result)}")

records_loaded:      0
current_watermark:   None
watermark_committed: False
bool(result):        False


## Third run — insert new data and re-run

New rows (id=4, 5) arrive. The next incremental load picks up only those.

In [6]:
engine = create_engine(sqlite_dsn)
with Session(engine) as session:
    session.add_all([
        Event(id=4, event_date=dt.date(2026, 5, 4), status="active"),
        Event(id=5, event_date=dt.date(2026, 5, 5), status="inactive"),
    ])
    session.commit()
engine.dispose()

with DataHelper(
    backend="sqlalchemy",
    connection_url=sqlite_dsn,
    poolclass="sqlalchemy.pool.NullPool",
    query_only=False,
    table="events",
) as helper:
    result = helper.pandas.load_incremental(
        watermark_field="event_date",
        watermark_source="events_table",
        watermark_store=watermark_store,
    )

print(f"records_loaded:      {result.records_loaded}")
print(f"previous_watermark:  {result.previous_watermark}")
print(f"current_watermark:   {result.current_watermark}")
print(f"watermark_committed: {result.watermark_committed}")
result.frame

records_loaded:      2
previous_watermark:  2026-05-03 00:00:00+00:00
current_watermark:   2026-05-05 00:00:00+00:00
watermark_committed: True


,id,event_date,status
0,4,2026-05-04 00:00:00+00:00,active
1,5,2026-05-05 00:00:00+00:00,inactive


## Async variant

`aload_incremental` works identically but returns an awaitable.

In [7]:

async def demo_async():
    async with DataHelper(
        backend="sqlalchemy",
        connection_url=sqlite_dsn,
        poolclass="sqlalchemy.pool.NullPool",
        query_only=False,
        table="events",
    ) as helper:
        result = await helper.pandas.aload_incremental(
            watermark_field="event_date",
            watermark_source="events_table",
            watermark_store=watermark_store,
        )
        return result

result = await demo_async()
print(f"async — records_loaded: {result.records_loaded}, committed: {result.watermark_committed}")

async — records_loaded: 0, committed: False


## Dask engine view

The `.dask` engine view supports the same `load_incremental()` API.

In [8]:
with DataHelper(
    backend="sqlalchemy",
    connection_url=sqlite_dsn,
    poolclass="sqlalchemy.pool.NullPool",
    query_only=False,
    table="events",
) as helper:
    result = helper.dask.load_incremental(
        watermark_field="event_date",
        watermark_source="events_table",
        watermark_store=watermark_store,
    )

print(f"dask — records_loaded: {result.records_loaded}")
result.frame

/Users/lvalverdeb/TeamDev/repo-split/boti-data/src/boti_data/db/partitioned_loader.py:36: UserWarning: Distributed SQL task payloads will carry the raw DSN credential because 'worker_connection_env_var' is not set on SqlDatabaseConfig. Set 'worker_connection_env_var' to the name of an environment variable that resolves the DSN on each worker to avoid serializing credentials.
  self._worker_config = WorkerSqlConfig.from_database_config(config)
WorkerSqlConfig created with raw DSN fallback; set worker_connection_env_var to avoid credential serialization.


dask — records_loaded: 0


,id,event_date,status
npartitions=1,,,
,Int64,"datetime64[ns, UTC]",string
,...,...,...


## Persisted state

The watermark file is a plain JSON document — easy to inspect, backup, or migrate.

In [9]:
print(watermark_path.read_text())

{"events_table": "2026-05-05 00:00:00+00:00"}


## Cleanup

In [10]:
tmp.cleanup()
print("Cleaned up.")

Cleaned up.
